# Spectral theorem

_Linear Algebra_

**Every symmetric matrix is a rotation, a scaling along perpendicular axes, then the rotation back.**

The spectral theorem says any symmetric matrix $A$ can be written $A = U\Lambda U^\top$.

     Here $\Lambda$ is diagonal (the eigenvalues) and $U$ holds the eigenvectors as columns — and those eigenvectors are perpendicular (orthonormal).

---

This notebook builds the idea up one step at a time: first make a symmetric matrix, then extract its eigenvalues and orthonormal eigenvectors, then reconstruct the matrix from those pieces — and finally see what the eigen-decomposition reveals about real data. Run each cell and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

We demonstrate $A = U\Lambda U^\top$ in three stages: (1) build a symmetric matrix, (2) decompose it into eigenvalues and orthonormal eigenvectors, (3) rebuild the original matrix from those pieces to prove the factorization holds.

### Step 1 — Build a symmetric matrix

The spectral theorem applies to symmetric matrices, so we first manufacture one. Starting from a random square matrix `B`, the sum `B + B^T` is always symmetric. We confirm it equals its own transpose before going further.

In [ ]:
rng = np.random.default_rng(0)

# B + B^T is always symmetric.
B = rng.standard_normal((3, 3))
A = B + B.T

is_symmetric = np.allclose(A, A.T)
print("symmetric:", is_symmetric)

### Step 2 — Extract eigenvalues and orthonormal eigenvectors

`np.linalg.eigh` is the solver for symmetric matrices: it returns real eigenvalues `w` and eigenvectors `Q` whose columns are **orthonormal**. Orthonormal means $Q^\top Q = I$ — the eigenvectors are mutually perpendicular unit vectors, which is the rotation `U` in the theorem.

In [ ]:
# eigh returns real eigenvalues w and orthonormal eigenvectors Q.
w, Q = np.linalg.eigh(A)
print("eigenvalues:", np.round(w, 4))

# Q is orthogonal: Q^T Q should equal the identity.
is_orthogonal = np.allclose(Q.T @ Q, np.eye(3))
print("Q^T Q == I:", is_orthogonal)

### Step 3 — Reconstruct the matrix from its pieces

The theorem claims $A = Q\,\Lambda\,Q^\top$, where $\Lambda$ is the diagonal matrix of eigenvalues. We assemble that product and check it matches the original `A`. Reading right to left, this is: rotate into the eigenvector frame, scale along each axis, then rotate back.

In [ ]:
# Reconstruct A = Q diag(w) Q^T and compare to the original.
Lambda = np.diag(w)
A_rebuilt = Q @ Lambda @ Q.T

matches = np.allclose(A, A_rebuilt)
print("A == Q L Q^T:", matches)

## Visualize the data & results

_What do the eigenvalues and principal axes of a symmetric matrix tell us about data?_

### Step 1 — Eigenvalues of a random symmetric matrix

First we look at a plain symmetric matrix `S = B + B^T`. Because it is only symmetric (not built as $A^\top A$), its eigenvalues are real but can be **positive or negative**. We grab those eigenvalues now and will plot their signs in the final step.

In [ ]:
rng = np.random.default_rng(0)

# Random symmetric matrix; eigenvalues are real but may be negative.
B = rng.standard_normal((3, 3))
S = B + B.T
eigs = np.linalg.eigvalsh(S)

### Step 2 — Find the principal axes of a 2D cloud

Now we apply the same eigen-decomposition to a *covariance* matrix. We generate a stretched, tilted cloud of 200 points, compute its covariance, and take its eigenvectors — these are the **principal axes** of the data. We sort them so the largest-variance direction comes first.

In [ ]:
# A stretched, tilted cloud of 2D points.
X = rng.standard_normal((200, 2)) @ np.array([[2, 1], [0, 0.5]])

# Covariance matrix is symmetric -> eigh gives its principal axes.
cov = np.cov(X.T)
w, V = np.linalg.eigh(cov)

# Sort so the largest-variance axis comes first.
order = np.argsort(w)[::-1]
w = w[order]
V = V[:, order]

### Step 3 — Plot the spectrum and the principal axes

Left: the symmetric matrix's eigenvalues, with negative bars colored red to show a general symmetric matrix is *not* necessarily PSD. Right: the data cloud with its two principal axes drawn through it, each scaled by $2\sqrt{\lambda}$ so the longer arrow marks the direction of greatest spread.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left — eigenvalues of S; red where negative.
colors = ['#ff7b72' if v < 0 else '#7ee787' for v in eigs]
ax1.bar(['lambda 1', 'lambda 2', 'lambda 3'], eigs, color=colors)
ax1.set_title('eigenvalues of symmetric S')

# Right — the data cloud with its principal axes overlaid.
ax2.scatter(X[:, 0], X[:, 1], color='#4ea1ff', s=10)
for k, c in zip(range(2), ['#ffb454', '#c89bff']):
    axis = V[:, k] * 2 * np.sqrt(w[k])      # scale each axis by 2*sqrt(eigenvalue)
    ax2.plot([-axis[0], axis[0]], [-axis[1], axis[1]], color=c, linewidth=2)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('principal axes')

plt.tight_layout()
plt.show()